In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

PARAMS = ["pH", "DO", "BOD", "COD", "TDS", "SS", "EC", "Temperature"]
TARGET = "WQI_Status"

df = pd.read_csv("surma_water_quality.csv")

df[PARAMS] = df[PARAMS].fillna(df[PARAMS].median())
imputer = KNNImputer(n_neighbors=5)
df[PARAMS] = imputer.fit_transform(df[PARAMS])

df = df.drop_duplicates().reset_index(drop=True)

scaler = StandardScaler()
df[PARAMS] = scaler.fit_transform(df[PARAMS])

X = df[PARAMS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y.value_counts())


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    "LogisticRegression": LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, class_weight="balanced", random_state=42),
}

rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    pro = clf.predict_proba(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1": f1_score(y_test, pred, average="macro", zero_division=0),
        "ROC_AUC": roc_auc_score(pd.get_dummies(y_test), pro, multi_class="ovr", average="macro"),
    })

results = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
results.to_csv("surma_model_results.csv", index=False)
results


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"acc": "accuracy", "prec": "precision_macro", "rec": "recall_macro", "f1": "f1_macro"}

cv_rows = []
for name, clf in models.items():
    out = cross_validate(clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "Model": name,
        "Accuracy": f"{out['test_acc'].mean()*100:.2f} +/- {out['test_acc'].std()*100:.1f}",
        "Precision": f"{out['test_prec'].mean()*100:.2f} +/- {out['test_prec'].std()*100:.1f}",
        "Recall": f"{out['test_rec'].mean()*100:.2f} +/- {out['test_rec'].std()*100:.1f}",
        "F1": f"{out['test_f1'].mean()*100:.2f} +/- {out['test_f1'].std()*100:.1f}",
    })

cv_df = pd.DataFrame(cv_rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
cv_df.to_csv("surma_cv_results.csv", index=False)
cv_df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

clf = LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
pro = clf.predict_proba(X_test)
classes = clf.classes_

print(classification_report(y_test, pred, digits=4))

cm = confusion_matrix(y_test, pred, labels=classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("surma_confusion_matrix.png", dpi=150)

fig, axes = plt.subplots(1, len(classes), figsize=(4 * len(classes), 4))
for ax, c in zip(np.atleast_1d(axes), classes):
    fpr, tpr, _ = roc_curve((y_test == c).astype(int), pro[:, list(classes).index(c)])
    ax.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_title(f"ROC: {c}")
    ax.legend()
plt.tight_layout()
plt.savefig("surma_roc.png", dpi=150)

import shap
explainer = shap.TreeExplainer(clf)
sv = explainer.shap_values(X_train)
plt.figure()
shap.summary_plot(sv, X_train, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("surma_shap.png", dpi=150)
